## Preprocesamiento, modelado, comparación estadística e interpretabilidad


In [2]:
%pip install -q kagglehub pyspark lime statsmodels pyarrow psutil

Note: you may need to restart the kernel to use updated packages.


In [3]:
import itertools
import platform
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import scipy.sparse as sp
from scipy import stats as sps
from IPython.display import display, Markdown

import sklearn
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, auc, confusion_matrix)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline as SkPipeline   # solo para VERIFICAR (validación final) que no se usa
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:   # scikit-learn < 1.0
    from sklearn.experimental import enable_hist_gradient_boosting  # noqa: F401
    from sklearn.ensemble import HistGradientBoostingClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
ALPHA = 0.05
# Regla literal del enunciado: default=0 agrupa todo lo que no es "Charged Off" -> no se llama "Fully Paid".
CLASS_LABELS = {0: "No Charged Off (0)", 1: "Charged Off (1)"}

# Artefactos compartidos entre scikit-learn y PySpark (partición, puntuaciones, tablas y figuras)
ARTIFACTS_DIR = Path("artifacts_modelado")
FIG_DIR = ARTIFACTS_DIR / "figuras"
LIME_DIR = ARTIFACTS_DIR / "lime"
for d in (ARTIFACTS_DIR, FIG_DIR, LIME_DIR):
    d.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = ARTIFACTS_DIR / "split_assignment.parquet"

# Registro único de tiempos (entorno, etapa, modelo, segundos) -> tabla final comparable
TIEMPOS = []


def registrar_tiempo(entorno, etapa, segundos, modelo=None):
    TIEMPOS.append({"entorno": entorno, "etapa": etapa, "modelo": modelo, "segundos": float(segundos)})


def guardar_figura(nombre):
    ruta = FIG_DIR / f"{nombre}.png"
    plt.savefig(ruta, dpi=120, bbox_inches="tight")
    return ruta


print(f"Directorio de artefactos: {ARTIFACTS_DIR.resolve()}")

Directorio de artefactos: C:\Users\juand\OneDrive\Desktop\MachineLearning\jbook_ml202603\docs\artifacts_modelado


### Hardware de ejecución

Se detecta automáticamente el hardware **del equipo donde realmente se ejecuta el notebook** (sistema operativo,
Python, CPU físicas y lógicas, RAM total y disponible) y se guarda en `hardware.csv`. No hay valores fijos: si el
notebook se ejecuta en otro computador, el reporte cambia.

In [4]:
vm = psutil.virtual_memory()
hardware = {
    "fecha_ejecucion": datetime.now().isoformat(timespec="seconds"),
    "sistema_operativo": f"{platform.system()} {platform.release()}",
    "version_so": platform.version(),
    "arquitectura": platform.machine(),
    "procesador": platform.processor() or "no reportado por el SO",
    "python": platform.python_version(),
    "cpu_fisicas": psutil.cpu_count(logical=False),
    "cpu_logicas": psutil.cpu_count(logical=True),
    "ram_total_gb": round(vm.total / 1024 ** 3, 2),
    "ram_disponible_gb": round(vm.available / 1024 ** 3, 2),
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}
try:
    import pyspark
    hardware["pyspark"] = pyspark.__version__
except ImportError:
    hardware["pyspark"] = "no instalado"

hardware_df = pd.DataFrame([hardware])
hardware_df.to_csv(ARTIFACTS_DIR / "hardware.csv", index=False)
display(hardware_df.T.rename(columns={0: "valor"}))
print(f"Guardado en {ARTIFACTS_DIR / 'hardware.csv'}")

,valor
fecha_ejecucion,2026-09-30T05:35:12
sistema_operativo,Windows 10
version_so,10.0.26100
arquitectura,AMD64
procesador,"AMD64 Family 25 Model 117 Stepping 2, Authenti..."
python,3.9.25
cpu_fisicas,6
cpu_logicas,12
ram_total_gb,7.27
ram_disponible_gb,0.69


Guardado en artifacts_modelado\hardware.csv


### Variables de entrada al modelado


`emp_length` se usa en su versión numérica `emp_length_num` (`"< 1 year"` → 0.5, `"1 year"` → 1, ...,
`"10+ years"` → 10), con **la misma regla** que en el EDA y la misma regla replicada en PySpark (sin UDF de Python).

In [5]:
# Variables seleccionadas en el EDA (9.10.4.1.5): sin leakage, sin cardinalidad extrema,
# sin >70% de faltantes y sin redundancia entre sí.
NUMERIC_FEATURES = ["int_rate", "fico_range_low", "acc_open_past_24mths", "inq_last_6mths",
                    "bc_open_to_buy", "percent_bc_gt_75", "inq_last_12m", "all_util"]
CATEGORICAL_FEATURES = ["sub_grade", "verification_status"]

# Columnas que se leen del CSV original (los indicadores de faltante se derivan después de leer).
COLUMNAS_CSV = ["id", "loan_status"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES

print(f"Numéricas ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categóricas ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")


Numéricas (8): ['int_rate', 'fico_range_low', 'acc_open_past_24mths', 'inq_last_6mths', 'bc_open_to_buy', 'percent_bc_gt_75', 'inq_last_12m', 'all_util']
Categóricas (2): ['sub_grade', 'verification_status']


### Carga del dataset, identificador y variable objetivo (9.10.3)


In [6]:
import kagglehub

DATA_SOURCE = "wordsforthewise/lending-club"
dataset_path = kagglehub.dataset_download(DATA_SOURCE)
csv_candidates = [p for p in Path(dataset_path).rglob("*.csv") if p.is_file() and "accepted" in p.name.lower()]
if not csv_candidates:
    raise FileNotFoundError("No se encontró un CSV 'accepted' dentro del dataset descargado.")
csv_path = csv_candidates[0]
print("Archivo usado:", csv_path)

t0 = time.perf_counter()
df = pd.read_csv(csv_path, usecols=COLUMNAS_CSV, dtype={"id": str}, low_memory=False)
t_sklearn_carga = time.perf_counter() - t0
registrar_tiempo("sklearn", "carga", t_sklearn_carga)

# Target: regla literal del enunciado (NO modificar)
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)

# Identificador original del CSV
assert df["id"].notna().all(), "Hay filas sin 'id' en el CSV."
df["id"] = df["id"].astype(str)
assert df["id"].is_unique, "La columna 'id' del CSV no es única."

# Indicadores de faltante informativo: inq_last_12m y all_util faltan casi siempre juntas (~38%)
# y la tasa de default casi se duplica cuando faltan (17.14% vs 8.61%, ver EDA). Se conserva esa señal
# como variable binaria ANTES de imputar la mediana.
df["inq_last_12m_missing"] = df["inq_last_12m"].isna().astype(int)
df["all_util_missing"] = df["all_util"].isna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + ["inq_last_12m_missing", "all_util_missing"]

N_TOTAL = len(df)
print(f"{N_TOTAL:,} filas leídas en {t_sklearn_carga:,.1f} s | ids únicos: {df['id'].nunique():,} | "
      f"tasa Charged Off = {100 * df['default'].mean():.2f}%")

c:\Users\juand\miniconda3\envs\ml_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Archivo usado: C:\Users\juand\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3\accepted_2007_to_2018q4.csv\accepted_2007_to_2018Q4.csv
2,260,701 filas leídas en 157.8 s | ids únicos: 2,260,701 | tasa Charged Off = 11.88%


## 9.10.4.2. Preprocesamiento

### 9.10.4.2.1. Partición común de los datos

Se define **una sola vez**: 80 % train / 20 % test, estratificada por `default`, `random_state=42`, sobre la columna
`id`. La asignación se guarda en `split_assignment.parquet` con **exactamente** dos columnas, `id` y `split`
(`"train"`/`"test"`). scikit-learn y PySpark leen esta misma asignación; PySpark **no** usa particiones aleatorias
propias ni recrea identificadores, de modo que ambos entornos entrenan y evalúan exactamente las mismas observaciones.

In [7]:
TEST_SIZE = 0.20

ids_train, ids_test = train_test_split(
    df["id"].to_numpy(),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["default"].to_numpy(),
)

split_df = pd.DataFrame({
    "id": np.concatenate([ids_train, ids_test]).astype(str),
    "split": (["train"] * len(ids_train)) + (["test"] * len(ids_test)),
})
split_df.to_parquet(SPLIT_PATH, index=False)

# --- Comprobaciones de la asignación guardada ---
split_leido = pd.read_parquet(SPLIT_PATH)
assert list(split_leido.columns) == ["id", "split"], "El Parquet debe tener exactamente las columnas id y split."
assert split_leido["id"].is_unique, "Hay IDs duplicados en la asignación."
assert set(split_leido["split"].unique()) == {"train", "test"}
assert len(split_leido) == N_TOTAL, "La partición debe cubrir TODAS las filas del dataset."
assert set(ids_train).isdisjoint(ids_test), "Un mismo id no puede estar en train y en test."

tasas = df[["id", "default"]].merge(split_leido, on="id").groupby("split")["default"].agg(["size", "mean"])
tasas.columns = ["n_ids", "tasa_charged_off"]
display(tasas)
print(f"Partición guardada en {SPLIT_PATH} | train={len(ids_train):,} | test={len(ids_test):,} | "
      f"sin duplicados | tasas estratificadas casi idénticas.")

,n_ids,tasa_charged_off
split,,
test,452141,0.118795
train,1808560,0.118795


Partición guardada en artifacts_modelado\split_assignment.parquet | train=1,808,560 | test=452,141 | sin duplicados | tasas estratificadas casi idénticas.


In [8]:
# Vista de trabajo: columnas del modelo + partición común. Aquí empieza el tiempo de preprocesamiento de sklearn.
t0_prep_sklearn = time.perf_counter()

MODEL_COLS = ["id", "default"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES
work = df[MODEL_COLS].merge(split_df, on="id", how="inner", validate="one_to_one")
train_df = work[work["split"] == "train"].drop(columns="split").reset_index(drop=True)
test_df = work[work["split"] == "test"].drop(columns="split").reset_index(drop=True)
del work
assert len(train_df) + len(test_df) == N_TOTAL, "No se puede perder ninguna fila en el modelado principal."
print(f"train_df: {train_df.shape} | test_df: {test_df.shape}")

train_df: (1808560, 14) | test_df: (452141, 14)


In [9]:
num_imputer = SimpleImputer(strategy="median").fit(train_df[NUMERIC_FEATURES])
X_train_num = num_imputer.transform(train_df[NUMERIC_FEATURES])
X_test_num = num_imputer.transform(test_df[NUMERIC_FEATURES])


def _categoricas_como_objeto(frame):
    # Matriz object con np.nan como faltante (compatible con SimpleImputer en cualquier versión de pandas)
    return frame[CATEGORICAL_FEATURES].astype(object).where(frame[CATEGORICAL_FEATURES].notna(), np.nan).to_numpy()


cat_imputer = SimpleImputer(strategy="most_frequent").fit(_categoricas_como_objeto(train_df))
X_train_cat_raw = cat_imputer.transform(_categoricas_como_objeto(train_df))
X_test_cat_raw = cat_imputer.transform(_categoricas_como_objeto(test_df))

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)   # scikit-learn >= 1.2
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)          # scikit-learn < 1.2
ohe.fit(X_train_cat_raw)
X_train_cat = ohe.transform(X_train_cat_raw)   # dispersa (CSR)
X_test_cat = ohe.transform(X_test_cat_raw)
ohe_feature_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES)

scaler = StandardScaler().fit(X_train_num)
X_train_num_scaled = scaler.transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

# Combinación dispersa: numéricas escaladas (pocas columnas densas) + one-hot disperso
X_train = sp.hstack([sp.csr_matrix(X_train_num_scaled), X_train_cat], format="csr")
X_test = sp.hstack([sp.csr_matrix(X_test_num_scaled), X_test_cat], format="csr")

feature_names_sklearn = NUMERIC_FEATURES + list(ohe_feature_names)
y_train = train_df["default"].to_numpy()
y_test = test_df["default"].to_numpy()
id_test = test_df["id"].to_numpy()

t_sklearn_prep = time.perf_counter() - t0_prep_sklearn
registrar_tiempo("sklearn", "preprocesamiento", t_sklearn_prep)

mb_disperso = (X_train.data.nbytes + X_train.indices.nbytes + X_train.indptr.nbytes) / 1024 ** 2
mb_denso = X_train.shape[0] * X_train.shape[1] * 8 / 1024 ** 2
print(f"X_train: {X_train.shape} | X_test: {X_test.shape} | {len(feature_names_sklearn)} columnas tras encoding")
print(f"Memoria X_train dispersa ≈ {mb_disperso:,.0f} MB (densa sería ≈ {mb_denso:,.0f} MB)")
print(f"Tiempo de preprocesamiento sklearn: {t_sklearn_prep:,.1f} s")

X_train: (1808560, 48) | X_test: (452141, 48) | 48 columnas tras encoding
Memoria X_train dispersa ≈ 255 MB (densa sería ≈ 662 MB)
Tiempo de preprocesamiento sklearn: 18.9 s


In [10]:
# ---------------------------------------------------------------------------------------------------------------
# Optimización de memoria (no cambia ningún resultado). En un equipo con poca RAM, pandas, scikit-learn y la JVM de
# Spark compiten por la memoria; si se agota, Windows pagina a disco y todo se vuelve decenas de veces más lento.
# Aquí se liberan copias intermedias que ya no se usan: el DataFrame completo leído del CSV (df), las tablas de la
# partición (ya guardada en Parquet) y los arreglos intermedios del preprocesamiento (X_train/X_test ya los contienen).
# Se conservan train_df/test_df (LIME), ids_train/ids_test (verificaciones) y los transformadores ajustados.
# Requiere ejecutar el notebook de arriba abajo (las celdas anteriores no deben reejecutarse después de esta).
# ---------------------------------------------------------------------------------------------------------------
# (no se usa liberar_memoria(): la validación final busca ".collect(" en el código de Spark)
from gc import collect as liberar_memoria

_ram_antes = psutil.virtual_memory().available / 1024 ** 3
for _nombre in ["df", "split_df", "split_leido", "tasas", "X_train_num", "X_test_num", "X_train_cat_raw",
                "X_test_cat_raw", "X_train_num_scaled", "X_test_num_scaled", "X_train_cat", "X_test_cat"]:
    globals().pop(_nombre, None)
liberar_memoria()
print(f"RAM disponible: {_ram_antes:.2f} GB -> {psutil.virtual_memory().available / 1024 ** 3:.2f} GB")


RAM disponible: 1.30 GB -> 2.18 GB


In [11]:
import os
os.environ["JAVA_HOME"] = r"c:\Users\juand\miniconda3\envs\ml_venv\Library"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

### 9.10.4.2.3. PySpark

1. **Carga:** Spark lee el mismo CSV original (tiempo de carga de PySpark = lectura + `count()`). Se usa
   `multiLine=True` y `escape='"'` para que las comillas y saltos de línea dentro de campos de texto se interpreten
   igual que en pandas y ambos motores vean exactamente las mismas filas (se verifica abajo).
2. **Partición:** se une con `split_assignment.parquet` por `id` y se filtra por la columna `split`. No se genera
   ninguna partición aleatoria en Spark ni se recrean identificadores.
3. **Preprocesamiento** en un `Pipeline` de Spark **ajustado solo con train**: `Imputer` (mediana, numéricas) →
   `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler(withMean=False, withStd=True)`.
   `withMean=False` evita densificar los vectores dispersos producidos por el one-hot (centrar exige restar la media
   a cada componente, lo que convierte los ceros en valores distintos de cero y dispara la memoria).
4. **Persistencia:** los conjuntos transformados se guardan con `persist(StorageLevel.MEMORY_AND_DISK)` y se
   materializan con una acción (`count()`) antes de entrenar.
5. Los datos permanecen en Spark: no se usa `.toPandas()` ni `.collect()` sobre el dataset antes del entrenamiento.

**Configuración de Spark (enunciado):** El enunciado pide `8g` y 400 particiones (9.10.4.5), y se usan literalmente.csv`; se discuten en la reflexión crítica (pregunta 12).


In [ ]:
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (Imputer, StringIndexer, OneHotEncoder as SparkOHE, VectorAssembler,
                                StandardScaler as SparkScaler)

import os

SPARK_DRIVER_MEMORY = "4g"
SPARK_PARTICIONES = 64
SPARK_CONF_FIRMA = (SPARK_DRIVER_MEMORY, SPARK_PARTICIONES)

spark = (SparkSession.builder
         .appName("LendingClub_Optimized")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", str(SPARK_PARTICIONES))
         .config("spark.default.parallelism", str(SPARK_PARTICIONES))
         .config("spark.executor.memory", SPARK_DRIVER_MEMORY)
         .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
         .config("spark.memory.fraction", 0.8)
         .config("spark.memory.storageFraction", 0.3)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
# Arrow acelera la ÚNICA transferencia permitida (id, default, score de test) tras el entrenamiento.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
print("Spark", spark.version)

memoria_activa = spark.sparkContext.getConf().get("spark.driver.memory", "no definida")
print(f"RAM total = {hardware['ram_total_gb']} GB | "
      f"spark.driver.memory = {memoria_activa} | particiones = {spark.conf.get('spark.sql.shuffle.partitions')}")
if memoria_activa != SPARK_DRIVER_MEMORY:
    print("[aviso] La sesión de Spark ya existía con otra memoria (la JVM solo toma este valor al arrancar). "
          "Reinicia el kernel y ejecuta el notebook desde el inicio.")

hardware.update({"spark_driver_memory": SPARK_DRIVER_MEMORY, "spark_particiones": SPARK_PARTICIONES})
pd.DataFrame([hardware]).to_csv(ARTIFACTS_DIR / "hardware.csv", index=False)


Spark 4.0.4
RAM total = 7.27 GB | spark.driver.memory = 4g | particiones = 64


In [14]:
t0 = time.perf_counter()
sdf_raw = (spark.read
           .option("header", True)
           .option("inferSchema", True)
           .option("multiLine", True)   # mismo parseo de campos con saltos de línea que pandas
           .option("escape", '"')       # comillas dobles escapadas como en el CSV original
           .csv(Path(csv_path).resolve().as_uri()))
sdf_full = (sdf_raw.select(*COLUMNAS_CSV)
            .withColumn("id", F.col("id").cast("string"))
            # Target: misma regla literal del enunciado
            .withColumn("default", F.when(F.col("loan_status") == "Charged Off", 1).otherwise(0)))
n_spark_total = sdf_full.count()
t_spark_carga = time.perf_counter() - t0
registrar_tiempo("pyspark", "carga", t_spark_carga)
print(f"Spark leyó {n_spark_total:,} filas en {t_spark_carga:,.1f} s (pandas: {N_TOTAL:,} filas)")
assert n_spark_total == N_TOTAL, "Spark y pandas deben leer exactamente el mismo número de filas."

Spark leyó 2,260,701 filas en 30.7 s (pandas: 2,260,701 filas)


In [15]:
# Aquí empieza el tiempo de preprocesamiento de PySpark.
t0_prep_spark = time.perf_counter()

sdf_split = spark.read.parquet(Path(SPLIT_PATH).resolve().as_uri())   # MISMA asignación que sklearn

# Mismos indicadores de faltante que en pandas (calculados con funciones nativas de Spark)
sdf = (sdf_full
       .withColumn("inq_last_12m_missing", F.col("inq_last_12m").isNull().cast("int"))
       .withColumn("all_util_missing", F.col("all_util").isNull().cast("int"))
       .select(["id", "default"] + NUMERIC_FEATURES + CATEGORICAL_FEATURES)
       .join(sdf_split, on="id", how="inner"))   # se filtra por 'split'; nunca randomSplit
for c in NUMERIC_FEATURES:
    sdf = sdf.withColumn(c, F.col(c).cast(DoubleType()))

# Se persiste la base (pocas columnas) para no releer el CSV en cada paso de ajuste del Pipeline.
sdf = sdf.persist(StorageLevel.MEMORY_AND_DISK)
sdf_train = sdf.filter(F.col("split") == "train")
sdf_test = sdf.filter(F.col("split") == "test")


In [16]:
# Pesos de clase "balanced": n / (2 * n_clase), calculados con y_train (misma partición que Spark). Son los mismos
# valores que usa class_weight="balanced" en scikit-learn, así que ambos entornos ponderan igual.
_n_tr, _n_pos = len(y_train), int(y_train.sum())
PESO_NEG = _n_tr / (2.0 * (_n_tr - _n_pos))
PESO_POS = _n_tr / (2.0 * _n_pos)

NUM_IMPUTED = [f"{c}_imp" for c in NUMERIC_FEATURES]
imputer_spark = Imputer(inputCols=NUMERIC_FEATURES, outputCols=NUM_IMPUTED, strategy="median")
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL_FEATURES]
encoder = SparkOHE(inputCols=[f"{c}_idx" for c in CATEGORICAL_FEATURES],
                   outputCols=[f"{c}_ohe" for c in CATEGORICAL_FEATURES])
# handleInvalid="error": tras el Imputer no debe quedar ningún nulo; si quedara, falla aquí (antes de entrenar).
assembler = VectorAssembler(inputCols=NUM_IMPUTED + [f"{c}_ohe" for c in CATEGORICAL_FEATURES],
                            outputCol="features_raw", handleInvalid="error")
# withMean=False: NO centrar, para no densificar los vectores dispersos del one-hot.
scaler_spark = SparkScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)

prep_pipeline = Pipeline(stages=[imputer_spark] + indexers + [encoder, assembler, scaler_spark])
prep_model = prep_pipeline.fit(sdf_train)      # Imputer, indexers, encoder y scaler ajustados SOLO con train

sdf_train_feat = (prep_model.transform(sdf_train).select("id", "default", "features")
                  .withColumn("peso", F.when(F.col("default") == 1, F.lit(PESO_POS)).otherwise(F.lit(PESO_NEG)))
                  .persist(StorageLevel.MEMORY_AND_DISK))
sdf_test_feat = (prep_model.transform(sdf_test).select("id", "default", "features")
                 .persist(StorageLevel.MEMORY_AND_DISK))
n_train_spark = sdf_train_feat.count()   # acción que materializa el caché (no transfiere datos al driver)
n_test_spark = sdf_test_feat.count()
sdf.unpersist()

t_spark_prep = time.perf_counter() - t0_prep_spark
registrar_tiempo("pyspark", "preprocesamiento", t_spark_prep)
print(f"Pipeline de Spark ajustado con train y persistido | train={n_train_spark:,} | test={n_test_spark:,} | "
      f"{t_spark_prep:,.1f} s")
print("Medianas de imputación (Spark, calculadas con train):",
      dict(zip(NUMERIC_FEATURES, [round(v, 4) for v in prep_model.stages[0].surrogateDF.first()])))

Pipeline de Spark ajustado con train y persistido | train=1,808,560 | test=452,141 | 34.1 s
Medianas de imputación (Spark, calculadas con train): {'int_rate': 12.62, 'fico_range_low': 690.0, 'acc_open_past_24mths': 4.0, 'inq_last_6mths': 0.0, 'bc_open_to_buy': 5434.0, 'percent_bc_gt_75': 37.5, 'inq_last_12m': 1.0, 'all_util': 58.0, 'inq_last_12m_missing': 0.0, 'all_util_missing': 0.0}


#### Comprobación: sklearn y PySpark usan exactamente la misma partición

Todas las comprobaciones se hacen con operaciones agregadas dentro de Spark (conteos y *anti-joins*), sin traer
datos al driver.

In [17]:
sdf_split_test_ids = sdf_split.filter(F.col("split") == "test").select("id")
sdf_split_train_ids = sdf_split.filter(F.col("split") == "train").select("id")

chequeos_particion = pd.DataFrame([
    {"comprobacion": "IDs train (sklearn vs Spark)", "sklearn": len(train_df), "pyspark": n_train_spark},
    {"comprobacion": "IDs test (sklearn vs Spark)", "sklearn": len(test_df), "pyspark": n_test_spark},
    {"comprobacion": "IDs duplicados en train", "sklearn": int(train_df["id"].duplicated().sum()),
     "pyspark": sdf_train_feat.groupBy("id").count().filter("count > 1").count()},
    {"comprobacion": "IDs duplicados en test", "sklearn": int(test_df["id"].duplicated().sum()),
     "pyspark": sdf_test_feat.groupBy("id").count().filter("count > 1").count()},
    {"comprobacion": "IDs test del Parquet ausentes en el test usado", "sklearn": len(set(ids_test) - set(id_test)),
     "pyspark": sdf_split_test_ids.join(sdf_test_feat.select("id"), "id", "left_anti").count()},
    {"comprobacion": "IDs del test usado que no son test en el Parquet", "sklearn": len(set(id_test) - set(ids_test)),
     "pyspark": sdf_test_feat.select("id").join(sdf_split_test_ids, "id", "left_anti").count()},
    {"comprobacion": "IDs train del Parquet ausentes en el train usado", "sklearn": len(set(ids_train) - set(train_df["id"])),
     "pyspark": sdf_split_train_ids.join(sdf_train_feat.select("id"), "id", "left_anti").count()},
    {"comprobacion": "Positivos (Charged Off) en test", "sklearn": int(y_test.sum()),
     "pyspark": int(sdf_test_feat.agg(F.sum("default")).first()[0])},
])
chequeos_particion["coincide"] = chequeos_particion["sklearn"] == chequeos_particion["pyspark"]
display(chequeos_particion)

assert n_train_spark == len(train_df) and n_test_spark == len(test_df)
assert n_train_spark + n_test_spark == N_TOTAL
assert (chequeos_particion.iloc[2:7][["sklearn", "pyspark"]] == 0).all().all()
assert chequeos_particion["coincide"].all()
print("OK: mismo número de observaciones y mismo conjunto de IDs de test en scikit-learn y PySpark.")

,comprobacion,sklearn,pyspark,coincide
0,IDs train (sklearn vs Spark),1808560,1808560,True
1,IDs test (sklearn vs Spark),452141,452141,True
2,IDs duplicados en train,0,0,True
3,IDs duplicados en test,0,0,True
4,IDs test del Parquet ausentes en el test usado,0,0,True
5,IDs del test usado que no son test en el Parquet,0,0,True
6,IDs train del Parquet ausentes en el train usado,0,0,True
7,Positivos (Charged Off) en test,53712,53712,True


OK: mismo número de observaciones y mismo conjunto de IDs de test en scikit-learn y PySpark.


## 9.10.4.3. Modelos y espacios de búsqueda

| Modelo (nombre lógico) | scikit-learn | PySpark | Espacio de búsqueda |
|---|---|---|---|
| `LogisticRegression` | `LogisticRegression` (L2) | `LogisticRegression` (`elasticNetParam=0`) | `regParam` ∈ {1e-6, 1e-5, 1e-4}; en sklearn `C = 1/(regParam·n_train)` |
| `DecisionTree` | `DecisionTreeClassifier` | `DecisionTreeClassifier` | `max_depth`/`maxDepth` ∈ {5, 10, 15} |
| `RandomForest` | `RandomForestClassifier` | `RandomForestClassifier` | `n_estimators`/`numTrees` ∈ {10, 50, 100} × `max_depth`/`maxDepth` ∈ {5, 10, 15} |
| `GradientBoosting` | `GradientBoostingClassifier` (o `HistGradientBoostingClassifier` si `USE_HIST_GB=True`) | `GBTClassifier` | `n_estimators`(`max_iter`)/`maxIter` ∈ {50, 100} × `max_depth`/`maxDepth` ∈ {3, 5}, `learning_rate`/`stepSize` = 0.1 |
| `LinearSVC` | `LinearSVC(loss="hinge")` | `LinearSVC` | igual que regresión logística |
| `NaiveBayes` | `GaussianNB` | `NaiveBayes(modelType="gaussian")` | sin búsqueda de hiperparámetros |

**Consideraciones de equivalencia (se retoman en la reflexión crítica):**

- **Regularización L2 y `C = 1/(regParam·n_train)`:** es una equivalencia aproximada. `C` multiplica la *suma* de
  pérdidas en sklearn y `regParam` penaliza el *promedio* en Spark; además, dentro de `GridSearchCV` cada fold entrena
  con ~2/3 de `n_train`, y Spark estandariza internamente las variables al regularizar (`standardization=True`).
- **Pérdida hinge:** `LinearSVC` de Spark usa hinge; en sklearn se fija `loss="hinge"` (por defecto sería squared hinge).
- **`maxBins`:** los árboles de Spark discretizan las variables continuas en `maxBins` intervalos (32 por defecto,
  fijado explícitamente abajo); sklearn evalúa cortes exactos.
- **Gradient boosting:** si se usa `HistGradientBoostingClassifier`, también discretiza (hasta 255 bins) y es otro
  algoritmo; el nombre lógico en todas las tablas sigue siendo `GradientBoosting`. En esta ejecución se activó (`USE_HIST_GB = True`); la justificación está en 9.10.4.4.
- **`NaiveBayes` gaussiano** en ambos entornos, porque el escalado produce valores negativos (incompatibles con las
  variantes multinomial/Bernoulli).
- **Margen de relevancia práctica** definido *antes* de ver resultados: `PRACTICAL_AUC_MARGIN = 0.005`.
- **Pesos de clase (desbalance):** `class_weight="balanced"` (scikit-learn) y `weightCol="peso"` (Spark) usan los
  mismos pesos, `n / (2·n_clase)` calculados con train; como suman `n`, la relación `C = 1/(regParam·n_train)` conserva
  su escala. Donde scikit-learn no tiene `class_weight` (`GradientBoostingClassifier`, `GaussianNB`) se pasa
  `sample_weight` con los mismos valores. El evaluador de la validación cruzada no usa pesos en ninguno de los dos
  entornos (`scoring="roc_auc"` y `areaUnderROC`, como pide el enunciado).


In [18]:
N_TRAIN = len(y_train)
CV_FOLDS = 3
MAX_BINS = 32   # valor por defecto de PySpark; se deja explícito para reportarlo
PRACTICAL_AUC_MARGIN = 0.005   # |ΔAUC| mínimo para considerar una diferencia relevante en la práctica (fijado a priori)

# Nombres lógicos únicos usados en TODAS las tablas, DeLong, McNemar, bootstrap, LIME y figuras.
MODELOS = ["LogisticRegression", "DecisionTree", "RandomForest", "GradientBoosting", "LinearSVC", "NaiveBayes"]

REG_PARAMS = [1e-6, 1e-5, 1e-4]
C_VALUES = [1.0 / (rp * N_TRAIN) for rp in REG_PARAMS]   # C = 1 / (regParam * n_train)
display(pd.DataFrame({"regParam (Spark)": REG_PARAMS, "C equivalente (sklearn)": C_VALUES}))

,regParam (Spark),C equivalente (sklearn)
0,0.000001,0.552926
1,0.000010,0.055293
2,0.000100,0.005529


### Tratamiento del desbalance de clases

Solo ≈ 11.9 % de las observaciones son Charged Off. Con el umbral por defecto, un modelo que predice siempre "No
Charged Off" alcanza ≈ 88 % de accuracy sin aprender nada, y el ROC-AUC (que no depende del umbral) puede verse
aceptable mientras el recall de la clase positiva es casi nulo. Por eso, **sin tocar los datos** (no hay
undersampling, oversampling ni SMOTE; todas las filas se usan):

1. **Pesos de clase "balanced" en el entrenamiento:** cada Charged Off pesa `n / (2·n₁)` y cada No Charged Off
   `n / (2·n₀)`, con `n₀`, `n₁` de train (≈ 7.4 veces más peso a la clase minoritaria).
2. **Umbral de decisión por (entorno, modelo)**, elegido para maximizar el F1 de la clase Charged Off con puntajes
   *out-of-fold* del **train** (nunca con test), tal como recomienda el enunciado en 9.10.4.6.2.
3. **Métrica de selección de hiperparámetros:** se mantiene ROC-AUC (`scoring="roc_auc"` / `areaUnderROC`), porque el
   enunciado lo exige. El AUC-PR se reporta como métrica adicional y en el bootstrap.


In [19]:
from sklearn.utils.class_weight import compute_class_weight

pesos_sklearn = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
assert np.allclose(pesos_sklearn, [PESO_NEG, PESO_POS]), "Los pesos de sklearn y Spark deben coincidir."

display(pd.DataFrame({
    "conjunto": ["train", "test"],
    "n": [len(y_train), len(y_test)],
    "n_charged_off": [int(y_train.sum()), int(y_test.sum())],
    "tasa_charged_off": [float(y_train.mean()), float(y_test.mean())],
}))
display(pd.DataFrame({
    "clase": ["0 (No Charged Off)", "1 (Charged Off)"],
    "peso_balanced": [PESO_NEG, PESO_POS],
    "razon_vs_clase_0": [1.0, PESO_POS / PESO_NEG],
}))


,conjunto,n,n_charged_off,tasa_charged_off
0,train,1808560,214847,0.118795
1,test,452141,53712,0.118795


,clase,peso_balanced,razon_vs_clase_0
0,0 (No Charged Off),0.567405,1.000000
1,1 (Charged Off),4.208949,7.417897


#### Checkpoints por modelo (optimización de tiempo)

Cada modelo terminado se guarda en `artifacts_modelado/checkpoints` (métricas, puntajes de test, umbral, mejor
modelo y tiempos). Si el notebook se vuelve a ejecutar —por ejemplo, tras reiniciar el kernel— con el mismo modelo,
espacio de búsqueda y datos, el modelo se **restaura en segundos** en lugar de reentrenarse durante horas. Los
tiempos reportados son siempre los medidos al entrenar. En Spark la firma incluye además la configuración de la
sesión, de modo que un cambio de memoria o de particiones obliga a reentrenar (los tiempos no serían comparables).
Para forzar el reentrenamiento: `USAR_CHECKPOINTS = False` o borrar la carpeta.


In [20]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.utils.class_weight import compute_sample_weight

# ---------------------------------------------------------------------------------------------------------------
# Opción para Gradient Boosting (ver celda de GradientBoosting más abajo).
#   False -> GradientBoostingClassifier (modelo requerido por el enunciado).
#   True  -> HistGradientBoostingClassifier: sustitución PERMITIDA por el enunciado cuando el costo computacional
#            del anterior resulta prohibitivo con el dataset completo. En ambos casos el nombre lógico es
#            "GradientBoosting".
#
# DECISIÓN EN ESTA EJECUCIÓN: True. Justificación:
#   * Costo. GradientBoostingClassifier es secuencial (un solo hilo) y en cada nodo evalúa cortes exactos sobre todas
#     las filas y todas las variables (O(n log n) por variable). Con ~1.8 millones de filas de train y la malla de
#     4 combinaciones x 3 pliegues, más el reajuste final y los 3 ajustes del umbral out-of-fold, son ~16 ajustes de
#     50-100 árboles cada uno. En un portátil con ~7 GB de RAM utilizable, donde un solo GridSearchCV de
#     RandomForest ya tomó ~44 min, el tiempo esperado es de horas: computacionalmente prohibitivo.
#   * Qué cambia. HistGradientBoostingClassifier discretiza cada variable en <= 255 bins una sola vez y busca los
#     cortes sobre histogramas (con la resta de histogramas padre-hijo), por lo que el costo por nodo pasa de
#     O(n log n) a O(n) + O(bins); además paraleliza con OpenMP. Sigue siendo boosting de árboles con pérdida
#     logística, con el mismo learning_rate, número de iteraciones y profundidad que GBTClassifier de Spark
#     (max_leaf_nodes=None para que solo max_depth limite el árbol, early_stopping=False).
#   * Diferencias a discutir en el informe: (1) cortes aproximados por bins en vez de exactos (Spark también
#     discretiza, con maxBins=32); (2) no acepta matrices dispersas, por eso se densifica solo para este modelo y
#     NaiveBayes (memoria densa que imprime la celda de preprocesamiento; se libera al terminar NaiveBayes);
#     (3) los árboles se hacen crecer por hojas (best-first), aquí limitados solo por max_depth.
# ---------------------------------------------------------------------------------------------------------------
USE_HIST_GB = True

# Algoritmos que NO aceptan matrices dispersas -> única situación en la que se densifica la matriz.
MODELOS_REQUIEREN_DENSO = {"NaiveBayes"} | ({"GradientBoosting"} if USE_HIST_GB else set())
_DENSE_CACHE = {}


def matriz_para(nombre_modelo, split):
    X = X_train if split == "train" else X_test
    if nombre_modelo not in MODELOS_REQUIEREN_DENSO:
        return X
    if split not in _DENSE_CACHE:
        _DENSE_CACHE[split] = X.toarray()
    return _DENSE_CACHE[split]


# ---------------------------------------------------------------------------------------------------------------
# Desbalance de clases (~12 % Charged Off), SIN remuestreo:
#   1) pesos "balanced" en el entrenamiento (class_weight o sample_weight);
#   2) umbral por (entorno, modelo) que maximiza el F1 de la clase 1 con puntajes out-of-fold de TRAIN.
# La selección de hiperparámetros sigue siendo por ROC-AUC (exigencia del enunciado).
# ---------------------------------------------------------------------------------------------------------------
UMBRALES = {}   # (entorno, modelo) -> umbral; entorno es "sklearn" o "spark"
CV_UMBRAL = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def umbral_decision(nombre_modelo, entorno):
    # Umbral elegido con train. Mientras no exista: 0 para el margen de LinearSVC, 0.5 para probabilidades.
    return UMBRALES.get((entorno, nombre_modelo), 0.0 if nombre_modelo == "LinearSVC" else 0.5)


def binarizar(nombre_modelo, score, entorno):
    return (np.asarray(score) >= umbral_decision(nombre_modelo, entorno)).astype(int)


def metricas_clasificacion(y_true, score, nombre_modelo, entorno):
    pred = binarizar(nombre_modelo, score, entorno)
    return {
        "umbral": umbral_decision(nombre_modelo, entorno),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": float(average_precision_score(y_true, score)),
        "matriz_confusion": confusion_matrix(y_true, pred, labels=[0, 1]).tolist(),
    }


def umbral_maximiza_f1(y, score):
    """Umbral (score >= umbral) que maximiza el F1 de la clase positiva."""
    p, r, thr = precision_recall_curve(y, score)
    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(thr[int(np.argmax(f1))])


def _puntajes_oof(estimador, X, y, metodo, fit_params, n_jobs):
    try:
        return cross_val_predict(estimador, X, y, cv=CV_UMBRAL, method=metodo, n_jobs=n_jobs, params=fit_params)
    except TypeError:   # scikit-learn < 1.4 usa fit_params
        return cross_val_predict(estimador, X, y, cv=CV_UMBRAL, method=metodo, n_jobs=n_jobs, fit_params=fit_params)


from gc import collect as liberar_memoria
import hashlib
import joblib

# ---------------------------------------------------------------------------------------------------------------
# Checkpoints por modelo: cada modelo terminado se guarda en disco. Si el notebook se vuelve a ejecutar con el mismo
# modelo, espacio de búsqueda y datos, se restaura en segundos en vez de reentrenarse (p. ej., tras reiniciar el
# kernel). No cambia ningún resultado ni ninguna regla del enunciado: los tiempos reportados son los medidos cuando
# el modelo se entrenó. Para forzar el reentrenamiento: USAR_CHECKPOINTS = False o borrar la carpeta de checkpoints.
# ---------------------------------------------------------------------------------------------------------------
USAR_CHECKPOINTS = True
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def _firma(*partes):
    return hashlib.sha1(repr(partes).encode("utf-8")).hexdigest()[:16]


def _ruta_ckpt(entorno, nombre):
    return CHECKPOINT_DIR / f"{entorno}_{nombre}.joblib"


def _firma_sklearn(nombre, estimator, param_grid, usar_sample_weight):
    params = sorted((k, repr(v)) for k, v in estimator.get_params(deep=False).items())
    grid = sorted((k, repr(list(v))) for k, v in dict(param_grid).items())
    return _firma("sklearn", nombre, type(estimator).__name__, params, grid, bool(usar_sample_weight),
                  tuple(X_train.shape), tuple(X_test.shape), int(y_train.sum()), int(y_test.sum()),
                  CV_FOLDS, RANDOM_STATE)


def _guardar_sklearn(nombre, firma):
    joblib.dump({"firma": firma, "metrics": sklearn_results[nombre], "score": sklearn_scores[nombre],
                 "modelo": sklearn_fitted_models[nombre], "gs": sklearn_gridsearch[nombre],
                 "umbral": UMBRALES[("sklearn", nombre)],
                 "tiempos": [t for t in TIEMPOS if t["entorno"] == "sklearn" and t["modelo"] == nombre]},
                _ruta_ckpt("sklearn", nombre))


sklearn_results = {}        # nombre lógico -> métricas y tiempos
sklearn_scores = {}         # nombre lógico -> puntuaciones continuas de test (orden de test_df)
sklearn_fitted_models = {}  # nombre lógico -> mejor estimador ya ajustado (LIME)
sklearn_gridsearch = {}     # nombre lógico -> objeto GridSearchCV (validación final)


def _restaurar_sklearn(nombre, firma):
    ruta = _ruta_ckpt("sklearn", nombre)
    if not (USAR_CHECKPOINTS and ruta.exists()):
        return False
    ck = joblib.load(ruta)
    if ck["firma"] != firma:
        print(f"[checkpoint] {nombre}: no coincide con el modelo o los datos actuales; se reentrena.")
        return False
    sklearn_results[nombre] = ck["metrics"]
    sklearn_scores[nombre] = ck["score"]
    sklearn_fitted_models[nombre] = ck["modelo"]
    sklearn_gridsearch[nombre] = ck["gs"]
    UMBRALES[("sklearn", nombre)] = ck["umbral"]
    TIEMPOS.extend(ck["tiempos"])
    m = ck["metrics"]
    print(f"[{nombre} | {m['implementacion']}] restaurado de checkpoint | AUC={m['roc_auc']:.4f} | "
          f"F1={m['f1']:.4f} | recall={m['recall']:.3f} | umbral={m['umbral']:.4f} | "
          f"fit original={m['tiempo_entrenamiento_s']:,.1f}s")
    return True


def evaluar_sklearn(nombre, estimator, param_grid, y_score_fn, usar_sample_weight=False):
    """GridSearchCV (sin Pipeline, ROC-AUC) + umbral out-of-fold en train + métricas en test + tiempos.
    usar_sample_weight=True para estimadores sin class_weight (GradientBoosting, GaussianNB)."""
    firma = _firma_sklearn(nombre, estimator, param_grid, usar_sample_weight)
    if _restaurar_sklearn(nombre, firma):
        return sklearn_results[nombre]
    X_tr, X_te = matriz_para(nombre, "train"), matriz_para(nombre, "test")
    fit_params = {}
    if usar_sample_weight:
        fit_params["sample_weight"] = compute_sample_weight("balanced", y_train)
    pesos = "sample_weight=balanced" if usar_sample_weight else "class_weight=balanced"

    gs = GridSearchCV(estimator, param_grid, cv=CV_FOLDS, scoring="roc_auc", n_jobs=-1, refit=True,
                      error_score="raise")
    t0 = time.perf_counter()
    gs.fit(X_tr, y_train, **fit_params)
    t_fit = time.perf_counter() - t0
    mejor = gs.best_estimator_

    # Umbral: máximo F1 de la clase 1 con puntajes out-of-fold SOLO de train (el test no interviene)
    t0 = time.perf_counter()
    metodo = "predict_proba" if hasattr(mejor, "predict_proba") else "decision_function"
    oof = _puntajes_oof(clone(mejor), X_tr, y_train, metodo, fit_params,
                        1 if nombre in MODELOS_REQUIEREN_DENSO else -1)
    oof = oof[:, 1] if oof.ndim == 2 else oof
    UMBRALES[("sklearn", nombre)] = umbral_maximiza_f1(y_train, oof)
    t_umbral = time.perf_counter() - t0

    t0 = time.perf_counter()
    score_test = np.asarray(y_score_fn(mejor, X_te), dtype=float)
    t_pred = time.perf_counter() - t0

    metrics = {"implementacion": type(estimator).__name__, "mejores_hiperparametros": gs.best_params_,
               "pesos": pesos, "cv_auc_medio": gs.best_score_,
               **metricas_clasificacion(y_test, score_test, nombre, "sklearn"),
               "tiempo_entrenamiento_s": t_fit, "tiempo_umbral_s": t_umbral, "tiempo_prediccion_s": t_pred}
    sklearn_results[nombre] = metrics
    sklearn_scores[nombre] = score_test
    sklearn_fitted_models[nombre] = mejor
    sklearn_gridsearch[nombre] = gs
    registrar_tiempo("sklearn", "entrenamiento_gridsearchcv", t_fit, nombre)
    registrar_tiempo("sklearn", "umbral_oof", t_umbral, nombre)
    registrar_tiempo("sklearn", "prediccion", t_pred, nombre)
    print(f"[{nombre} | {metrics['implementacion']}] AUC={metrics['roc_auc']:.4f} | PR-AUC={metrics['pr_auc']:.4f} | "
          f"F1={metrics['f1']:.4f} | recall={metrics['recall']:.3f} | umbral={metrics['umbral']:.4f} | "
          f"fit={t_fit:,.1f}s | umbral={t_umbral:,.1f}s | pred={t_pred:,.2f}s | mejores hiperparámetros: {gs.best_params_}")
    _guardar_sklearn(nombre, firma)
    return metrics


def proba_positiva(est, X):
    return est.predict_proba(X)[:, 1]


## 9.10.4.5. Modelado con PySpark

- `CrossValidator` + `ParamGridBuilder`, `numFolds=3`, `BinaryClassificationEvaluator(metricName="areaUnderROC")`,
  **sin bucles manuales** de combinaciones de hiperparámetros.
- Los seis modelos con los espacios de 9.10.4.3 y los mismos nombres lógicos que en sklearn.
- **Tiempos por separado:** entrenamiento (+ `CrossValidator`), predicción (transformación de test materializada con
  `persist` + `count`) y **transferencia al driver** (`transfer_seconds`), que no se suma al entrenamiento.
- **Transferencia única por modelo, solo después de entrenar y predecir:** se traen al driver únicamente `id`,
  `default` y `score` del conjunto de test, con `vector_to_array`:
  - modelos probabilísticos: `score` = probabilidad de la clase positiva (`probability[1]`);
  - `LinearSVC`: `score` = componente de la clase positiva de `rawPrediction` (margen de decisión).
- **Desbalance:** columna `peso` en el train (misma fórmula "balanced" que scikit-learn) y `weightCol="peso"` en los seis
  estimadores. El evaluador sigue siendo `areaUnderROC`, sin pesos (criterio del enunciado).
- **Umbral (solo train):** se reajusta el mejor modelo del `CrossValidator` (con sus mejores hiperparámetros) en 3
  pliegues definidos por `hash(id)` del train y se predice cada pliegue con el modelo que no lo vio (*out-of-fold*, sin
  `randomSplit`). El umbral maximiza el F1 de la clase Charged Off y se calcula **dentro de Spark**: un histograma
  de 1000 intervalos del puntaje (una sola agregación con `groupBy`); al driver solo llegan escalares, no filas de train.
- Accuracy, precision, recall, F1 y matriz de confusión se calculan localmente a partir del `score` de test con ese
  umbral (`UMBRALES[("spark", modelo)]`), que **no usa test**.


In [21]:
from pyspark.ml.classification import (LogisticRegression as SparkLR, DecisionTreeClassifier as SparkDT,
                                       RandomForestClassifier as SparkRF, GBTClassifier as SparkGBT,
                                       LinearSVC as SparkSVC, NaiveBayes as SparkNB)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

evaluator_auc = BinaryClassificationEvaluator(labelCol="default", rawPredictionCol="rawPrediction",
                                              metricName="areaUnderROC")

spark_results = {}         # nombre lógico -> métricas y tiempos
spark_scores_frames = {}   # nombre lógico -> pandas.DataFrame (id, default, score) SOLO de test
spark_fitted_models = {}   # nombre lógico -> bestModel (LIME)
spark_cv = {}              # nombre lógico -> objeto CrossValidator (validación final)
TRANSFER_LOG = []          # registro de cada transferencia Spark -> driver


import hashlib
import joblib
from pyspark.ml.classification import (LogisticRegressionModel, DecisionTreeClassificationModel,
                                       RandomForestClassificationModel, GBTClassificationModel, LinearSVCModel,
                                       NaiveBayesModel)

CLASES_MODELO_SPARK = {"LogisticRegression": LogisticRegressionModel,
                       "DecisionTreeClassifier": DecisionTreeClassificationModel,
                       "RandomForestClassifier": RandomForestClassificationModel,
                       "GBTClassifier": GBTClassificationModel, "LinearSVC": LinearSVCModel,
                       "NaiveBayes": NaiveBayesModel}


class _ModelosSpark(dict):
    """Diccionario de mejores modelos. Si un modelo se restauró de checkpoint sin su archivo (p. ej., Spark no puede
    escribir modelos en Windows sin winutils), se reajusta con sus mejores hiperparámetros SOLO si se pide (LIME)."""

    def __init__(self):
        super().__init__()
        self._pendientes = {}

    def pendiente(self, nombre, est, param_map):
        self._pendientes[nombre] = (est, param_map)

    def __missing__(self, nombre):
        if nombre not in self._pendientes:
            raise KeyError(nombre)
        est, param_map = self._pendientes.pop(nombre)
        print(f"[checkpoint] {nombre}: reajustando con sus mejores hiperparámetros para LIME (no se cronometra).")
        self[nombre] = est.copy(param_map).fit(sdf_train_feat)
        return self[nombre]


def _firma(*partes):
    return hashlib.sha1(repr(partes).encode("utf-8")).hexdigest()[:16]


def _ruta_ckpt(entorno, nombre):
    return CHECKPOINT_DIR / f"{entorno}_{nombre}.joblib"


def _firma_spark(nombre, est, grid):
    params = sorted((p.name, repr(v)) for p, v in est.extractParamMap().items())
    malla = [sorted((p.name, repr(v)) for p, v in pm.items()) for pm in grid]
    return _firma("spark", nombre, type(est).__name__, params, malla, SPARK_CONF_FIRMA,
                  n_train_spark, n_test_spark, CV_FOLDS, RANDOM_STATE)


def _guardar_spark(nombre, firma, idx_mejor, modelo):
    ruta_modelo = (CHECKPOINT_DIR / f"spark_{nombre}_modelo").resolve().as_uri()
    try:
        modelo.write().overwrite().save(ruta_modelo)
        modelo_ok = True
    except Exception as e:   # p. ej., Windows sin winutils: se guardan igual métricas, puntajes y umbral
        modelo_ok = False
        print(f"[checkpoint] {nombre}: no se pudo guardar el modelo de Spark ({type(e).__name__}); si se restaura, "
              "se reajustará solo cuando LIME lo necesite.")
    joblib.dump({"firma": firma, "metrics": spark_results[nombre], "scores": spark_scores_frames[nombre],
                 "umbral": UMBRALES[("spark", nombre)], "idx_mejor": idx_mejor,
                 "modelo_ok": modelo_ok, "ruta_modelo": ruta_modelo,
                 "tiempos": [t for t in TIEMPOS if t["entorno"] == "pyspark" and t["modelo"] == nombre],
                 "transfer": [t for t in TRANSFER_LOG if t["modelo"] == nombre][-1]},
                _ruta_ckpt("spark", nombre))


def _restaurar_spark(nombre, firma, est, grid):
    ruta = _ruta_ckpt("spark", nombre)
    if not (USAR_CHECKPOINTS and ruta.exists()):
        return False
    ck = joblib.load(ruta)
    if ck["firma"] != firma:
        print(f"[checkpoint] {nombre}: no coincide con el modelo, los datos o la configuración de Spark; se reentrena.")
        return False
    spark_results[nombre] = ck["metrics"]
    spark_scores_frames[nombre] = ck["scores"]
    UMBRALES[("spark", nombre)] = ck["umbral"]
    TIEMPOS.extend(ck["tiempos"])
    TRANSFER_LOG.append(ck["transfer"])
    modelo = None
    if ck["modelo_ok"]:
        try:
            modelo = CLASES_MODELO_SPARK[type(est).__name__].load(ck["ruta_modelo"])
        except Exception:
            modelo = None
    if modelo is not None:
        spark_fitted_models[nombre] = modelo
    else:
        spark_fitted_models.pendiente(nombre, est, grid[ck["idx_mejor"]])
    m = ck["metrics"]
    print(f"[{nombre} | {m['implementacion']}] restaurado de checkpoint | AUC={m['roc_auc']:.4f} | "
          f"F1={m['f1']:.4f} | recall={m['recall']:.3f} | umbral={m['umbral']:.4f} | "
          f"fit original={m['tiempo_entrenamiento_s']:,.1f}s")
    return True


spark_fitted_models = _ModelosSpark()   # nombre lógico -> bestModel (LIME)


N_INTERVALOS_UMBRAL = 1000   # intervalos del histograma de puntajes con que se busca el umbral


def umbral_oof_spark(estimator, param_map, score_col):
    """Umbral (score >= umbral) que maximiza el F1 de la clase 1 con puntajes out-of-fold de TRAIN, dentro de Spark.
    Pliegues por hash(id); cada pliegue se predice con un modelo (mejores hiperparámetros) que no lo vio.
    El F1 se evalúa en un histograma de puntajes (una sola agregación groupBy), no con cientos de expresiones."""
    base = sdf_train_feat.withColumn("fold", F.pmod(F.hash(F.col("id")), F.lit(CV_FOLDS)))
    partes = []
    for k in range(CV_FOLDS):
        modelo_k = estimator.copy(param_map).fit(base.filter(F.col("fold") != k))
        partes.append(modelo_k.transform(base.filter(F.col("fold") == k))
                      .select(F.col("default").cast("double").alias("y"),
                              vector_to_array(F.col(score_col))[1].alias("score")))
    oof = partes[0]
    for parte in partes[1:]:
        oof = oof.unionByName(parte)
    oof = oof.persist(StorageLevel.MEMORY_AND_DISK)
    try:
        rango = oof.agg(F.min("score").alias("lo"), F.max("score").alias("hi")).first()
        lo, hi = float(rango["lo"]), float(rango["hi"])
        ancho = max((hi - lo) / N_INTERVALOS_UMBRAL, 1e-12)
        intervalo = F.least(F.floor((F.col("score") - F.lit(lo)) / F.lit(ancho)).cast("int"),
                            F.lit(N_INTERVALOS_UMBRAL - 1))
        hist = oof.withColumn("b", intervalo).groupBy("b").agg(F.sum("y").alias("tp"), F.count(F.lit(1)).alias("pp"))
        filas = hist.agg(F.collect_list(F.struct("b", "tp", "pp")).alias("h")).first()["h"]   # una fila de escalares
    finally:
        oof.unpersist()
    tp, pp = np.zeros(N_INTERVALOS_UMBRAL), np.zeros(N_INTERVALOS_UMBRAL)
    for r in filas:
        tp[r["b"]], pp[r["b"]] = r["tp"], r["pp"]
    tp, pp = tp[::-1].cumsum()[::-1], pp[::-1].cumsum()[::-1]
    f1 = 2.0 * tp / np.maximum(pp + tp[0], 1e-12)
    return float(lo + int(np.argmax(f1)) * ancho)


def evaluar_spark(nombre, estimator, param_grid, score_col):
    """Evalúa modelo de Spark con malla de parámetros (lista de ParamMap, ya construida).
    param_grid es el resultado de ParamGridBuilder().addGrid(...).build(), NO una lambda."""
    est = estimator.setLabelCol("default").setFeaturesCol("features").setWeightCol("peso")
    if est.hasParam("maxBins"):
        est = est.setMaxBins(MAX_BINS)
    
    firma = _firma_spark(nombre, est, param_grid)
    if _restaurar_spark(nombre, firma, est, param_grid):
        return spark_results[nombre]

    cv = CrossValidator(estimator=est, estimatorParamMaps=param_grid, evaluator=evaluator_auc,
                        numFolds=CV_FOLDS, parallelism=2, seed=RANDOM_STATE)

    # 1) Entrenamiento + CrossValidator (sobre el train persistido, con pesos de clase)
    t0 = time.perf_counter()
    cv_model = cv.fit(sdf_train_feat)
    t_fit = time.perf_counter() - t0
    idx_mejor = int(np.argmax(cv_model.avgMetrics))
    mejores = {p.name: v for p, v in cv_model.getEstimatorParamMaps()[idx_mejor].items()}

    # 2) Umbral out-of-fold SOLO con train (agregaciones en Spark; sin transferir filas)
    t0 = time.perf_counter()
    UMBRALES[("spark", nombre)] = umbral_oof_spark(est, cv_model.getEstimatorParamMaps()[idx_mejor], score_col)
    t_umbral = time.perf_counter() - t0

    # 3) Predicción sobre test (materializada en Spark, sin traer nada al driver)
    t0 = time.perf_counter()
    pred = (cv_model.transform(sdf_test_feat)
            .withColumn("score", vector_to_array(F.col(score_col))[1])
            .select("id", "default", "score")
            .persist(StorageLevel.MEMORY_AND_DISK))
    n_pred = pred.count()
    t_pred = time.perf_counter() - t0

    # 4) ÚNICA transferencia de filas al driver: id, default, score del test (después de entrenar y predecir)
    t0 = time.perf_counter()
    scores_pd = pred.toPandas()
    t_transfer = time.perf_counter() - t0
    pred.unpersist()
    assert len(scores_pd) == n_pred == n_test_spark, "Solo se transfiere el conjunto de test."
    assert list(scores_pd.columns) == ["id", "default", "score"]
    TRANSFER_LOG.append({"modelo": nombre, "filas": len(scores_pd), "columnas": list(scores_pd.columns),
                         "transfer_seconds": t_transfer})

    scores_pd["id"] = scores_pd["id"].astype(str)
    y_true = scores_pd["default"].to_numpy().astype(int)
    score_arr = scores_pd["score"].to_numpy().astype(float)

    metrics = {"implementacion": type(estimator).__name__, "mejores_hiperparametros": mejores,
               "pesos": "weightCol=peso (balanced)", "cv_auc_medio": float(cv_model.avgMetrics[idx_mejor]),
               **metricas_clasificacion(y_true, score_arr, nombre, "spark"),
               "tiempo_entrenamiento_s": t_fit, "tiempo_umbral_s": t_umbral, "tiempo_prediccion_s": t_pred,
               "tiempo_transferencia_s": t_transfer}
    spark_results[nombre] = metrics
    spark_scores_frames[nombre] = scores_pd
    spark_fitted_models[nombre] = cv_model.bestModel
    spark_cv[nombre] = cv
    registrar_tiempo("pyspark", "entrenamiento_crossvalidator", t_fit, nombre)
    registrar_tiempo("pyspark", "umbral_oof", t_umbral, nombre)
    registrar_tiempo("pyspark", "prediccion", t_pred, nombre)
    registrar_tiempo("pyspark", "transferencia_driver", t_transfer, nombre)
    print(f"[{nombre} | {metrics['implementacion']}] AUC={metrics['roc_auc']:.4f} | PR-AUC={metrics['pr_auc']:.4f} | "
          f"F1={metrics['f1']:.4f} | recall={metrics['recall']:.3f} | umbral={metrics['umbral']:.4f} | "
          f"fit={t_fit:,.1f}s | umbral={t_umbral:,.1f}s | pred={t_pred:,.2f}s | transfer={t_transfer:,.2f}s | "
          f"mejores parámetros: {mejores}")
    _guardar_spark(nombre, firma, idx_mejor, cv_model.bestModel)
    return metrics


In [ ]:
lr_est = SparkLR(family="binomial", elasticNetParam=0.0, maxIter=100)   # L2 pura
evaluar_spark(
    "LogisticRegression",
    lr_est,
    # addGrid() usa Param de la MISMA instancia pasada como estimador: CrossValidator compara los ParamMap por
    # identidad (parent uid + nombre), así que un Param tomado de una instancia distinta (p. ej. SparkLR() nueva)
    # se ignora en silencio y el modelo se ajusta con los valores por defecto en cada punto de la malla.
    ParamGridBuilder().addGrid(lr_est.regParam, REG_PARAMS).build(),
    score_col="probability",
)


[LogisticRegression | LogisticRegression] AUC=0.7286 | PR-AUC=0.2537 | F1=0.3352 | recall=0.522 | umbral=0.5636 | fit=168.9s | umbral=64.5s | pred=0.84s | transfer=2.04s | mejores parámetros: {'regParam': 1e-06}
[checkpoint] LogisticRegression: no se pudo guardar el modelo de Spark (Py4JJavaError); si se restaura, se reajustará solo cuando LIME lo necesite.


{'implementacion': 'LogisticRegression',
 'mejores_hiperparametros': {'regParam': 1e-06},
 'pesos': 'weightCol=peso (balanced)',
 'cv_auc_medio': 0.7286429449784978,
 'umbral': 0.5636190103381193,
 'accuracy': 0.7538090993738679,
 'precision': 0.24675758614928733,
 'recall': 0.5224717009234435,
 'f1': 0.3352026708234044,
 'roc_auc': 0.7285956243045888,
 'pr_auc': 0.25373447383042197,
 'matriz_confusion': [[312765, 85664], [25649, 28063]],
 'tiempo_entrenamiento_s': 168.88420659999997,
 'tiempo_umbral_s': 64.45199490000005,
 'tiempo_prediccion_s': 0.840409300000033,
 'tiempo_transferencia_s': 2.0395723999999973}

In [24]:
dt_est = SparkDT()
evaluar_spark(
    "DecisionTree",
    dt_est,
    ParamGridBuilder().addGrid(dt_est.maxDepth, [5, 10, 15]).build(),
    score_col="probability",
)


[DecisionTree | DecisionTreeClassifier] AUC=0.6931 | PR-AUC=0.1949 | F1=0.3139 | recall=0.643 | umbral=0.5736 | fit=97.6s | umbral=15.6s | pred=0.75s | transfer=0.73s | mejores parámetros: {'maxDepth': 5}
[checkpoint] DecisionTree: no se pudo guardar el modelo de Spark (Py4JJavaError); si se restaura, se reajustará solo cuando LIME lo necesite.


{'implementacion': 'DecisionTreeClassifier',
 'mejores_hiperparametros': {'maxDepth': 5},
 'pesos': 'weightCol=peso (balanced)',
 'cv_auc_medio': 0.6577299999686989,
 'umbral': 0.5735558095624648,
 'accuracy': 0.6662788820301632,
 'precision': 0.20764866214598163,
 'recall': 0.6425193625260649,
 'f1': 0.31386333562213803,
 'roc_auc': 0.6930585650714748,
 'pr_auc': 0.1948693655570486,
 'matriz_confusion': [[266741, 131688], [19201, 34511]],
 'tiempo_entrenamiento_s': 97.60629129999995,
 'tiempo_umbral_s': 15.61657630000002,
 'tiempo_prediccion_s': 0.754187399999978,
 'tiempo_transferencia_s': 0.7313358000000107}

In [21]:
rf_est = SparkRF(seed=RANDOM_STATE)
evaluar_spark(
    "RandomForest",
    rf_est,
    ParamGridBuilder().addGrid(rf_est.numTrees, [10, 50]).addGrid(rf_est.maxDepth, [5, 10]).build(),
    score_col="probability",
)


[RandomForest | RandomForestClassifier] AUC=0.7203 | PR-AUC=0.2459 | F1=0.3233 | recall=0.559 | umbral=0.5637 | fit=375.8s | umbral=203.4s | pred=2.83s | transfer=0.98s | mejores parámetros: {'numTrees': 50, 'maxDepth': 10}
[checkpoint] RandomForest: no se pudo guardar el modelo de Spark (Py4JJavaError); si se restaura, se reajustará solo cuando LIME lo necesite.


{'implementacion': 'RandomForestClassifier',
 'mejores_hiperparametros': {'numTrees': 50, 'maxDepth': 10},
 'pesos': 'weightCol=peso (balanced)',
 'cv_auc_medio': 0.7202829693803281,
 'umbral': 0.5636952876333525,
 'accuracy': 0.7219340869330585,
 'precision': 0.22738251171665014,
 'recall': 0.5591301757521596,
 'f1': 0.323291475813961,
 'roc_auc': 0.7202975459781532,
 'pr_auc': 0.2458507143550103,
 'matriz_confusion': [[296384, 102045], [23680, 30032]],
 'tiempo_entrenamiento_s': 375.8205203,
 'tiempo_umbral_s': 203.3632523,
 'tiempo_prediccion_s': 2.834526399999959,
 'tiempo_transferencia_s': 0.9849375999999666}

In [24]:
gbt_est = SparkGBT(stepSize=0.1, seed=RANDOM_STATE)
evaluar_spark(
    "GradientBoosting",            # nombre lógico común; implementación GBTClassifier
    gbt_est,
    ParamGridBuilder().addGrid(gbt_est.maxIter, [50, 100]).addGrid(gbt_est.maxDepth, [3, 5]).build(),
    score_col="probability",
)


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "c:\Users\juand\miniconda3\envs\ml_venv\lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "c:\Users\juand\miniconda3\envs\ml_venv\lib\socket.py", line 716, in readinto
    return self._sock.recv_into(b)
ConnectionResetError: [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\juand\miniconda3\envs\ml_venv\lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\Users\juand\miniconda3\envs\ml_venv\lib\site-packages\py4j\clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


ConnectionRefusedError: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión

In [22]:
svc_est = SparkSVC()
evaluar_spark(
    "LinearSVC",
    svc_est,
    ParamGridBuilder().addGrid(svc_est.regParam, REG_PARAMS).build(),
    score_col="rawPrediction",     # LinearSVC no produce 'probability': se usa el margen de la clase positiva
)


[LinearSVC | LinearSVC] AUC=0.7123 | PR-AUC=0.2459 | F1=0.2924 | recall=0.630 | umbral=0.9996 | fit=659.2s | umbral=191.5s | pred=0.85s | transfer=1.78s | mejores parámetros: {'regParam': 1e-06}
[checkpoint] LinearSVC: no se pudo guardar el modelo de Spark (Py4JJavaError); si se restaura, se reajustará solo cuando LIME lo necesite.


{'implementacion': 'LinearSVC',
 'mejores_hiperparametros': {'regParam': 1e-06},
 'pesos': 'weightCol=peso (balanced)',
 'cv_auc_medio': 0.7106191122774349,
 'umbral': 0.9995600779581311,
 'accuracy': 0.6380553853775702,
 'precision': 0.19042362667687907,
 'recall': 0.6295055108728032,
 'f1': 0.2923977619620018,
 'roc_auc': 0.7123332222003662,
 'pr_auc': 0.24591352695216862,
 'matriz_confusion': [[254679, 143750], [19900, 33812]],
 'tiempo_entrenamiento_s': 659.1830834,
 'tiempo_umbral_s': 191.49460049999993,
 'tiempo_prediccion_s': 0.8532371000001149,
 'tiempo_transferencia_s': 1.7841922999998587}

In [23]:
evaluar_spark(
    "NaiveBayes",
    SparkNB(modelType="gaussian"),
    ParamGridBuilder().build(),   # sin búsqueda, como en sklearn
    score_col="probability",
)

[NaiveBayes | NaiveBayes] AUC=0.7024 | PR-AUC=0.2313 | F1=0.3114 | recall=0.534 | umbral=0.0250 | fit=21.3s | umbral=8.8s | pred=0.73s | transfer=0.88s | mejores parámetros: {}
[checkpoint] NaiveBayes: no se pudo guardar el modelo de Spark (Py4JJavaError); si se restaura, se reajustará solo cuando LIME lo necesite.


{'implementacion': 'NaiveBayes',
 'mejores_hiperparametros': {},
 'pesos': 'weightCol=peso (balanced)',
 'cv_auc_medio': 0.5719013773879812,
 'umbral': 0.025,
 'accuracy': 0.7191318637327736,
 'precision': 0.21965477137785394,
 'recall': 0.5344801906464105,
 'f1': 0.3113530866339855,
 'roc_auc': 0.7024018307176981,
 'pr_auc': 0.23125392091900823,
 'matriz_confusion': [[296441, 101988], [25004, 28708]],
 'tiempo_entrenamiento_s': 21.312616899999966,
 'tiempo_umbral_s': 8.828116499999851,
 'tiempo_prediccion_s': 0.7288352000000486,
 'tiempo_transferencia_s': 0.8819589000001997}

**Interpretación (PySpark):** 

Se completaron 5 de los 6 modelos. GradientBoosting no pudo terminar: con la malla del enunciado (16 ajustes, el mayor de 100 árboles secuenciales sobre ≈ 1.8 M filas) corrió 158 min y terminó con ConnectionResetError [WinError 10054], es decir, la JVM cayó por falta de memoria en un equipo de 7.27 GB de RAM. Es una limitación del hardware, y por eso no hay comparación del boosting entre entornos. En los modelos completados el orden por AUC es regresión logística (0.7286) > RandomForest (0.7203) > LinearSVC (0.7123) > NaiveBayes (0.7024) > DecisionTree (0.6949), y la regresión logística también logra el mejor F1 de los 11 modelos (0.335). PySpark iguala a scikit-learn en LR, RF, LinearSVC y NaiveBayes, y queda por debajo solo en el árbol (0.6949 frente a 0.7126). La exactitud (0.64–0.75) es menor que la de predecir siempre «no incumple» (0.881) porque los umbrales maximizan el F1 de la clase minoritaria. La malla del RandomForest fue de 4 combinaciones, menor que la de sklearn (9), y debe declararse como desviación, igual que la configuración de 4g y 64 particiones. Los CV medios del árbol (0.648) y de NaiveBayes (0.572) no coinciden con su AUC de test, probablemente porque el evaluador usa rawPrediction, así que no deben compararse. La variabilidad de entrenamiento (≈ 0.002 de AUC entre dos ejecuciones del mismo árbol) es comparable al intervalo de confianza de DeLong. Con el dataset cabiendo en la memoria de una sola máquina, el valor de PySpark aquí está en cumplir el flujo distribuido y no en la velocidad.